# 41 · CANDLE_SAVI · bulk_RNA_seq · differential expression between diagnoses

Reads `counts_raw.rds` and `samples.rds` (notebook 40). Writes, for each comparison, the full table
`de_<pair>.csv` and the top genes `top_genes_<pair>.csv` in `data/run_artifacts/CANDLE_SAVI/`.

**Counts.** The raw counts, not the batch-corrected ones: batch enters the model as a term, as the authors
of DESeq2 and limma advise (Love MI, Anders S, Kim V, Huber W. *F1000Research* 2015;4:1070).

**Samples.** The 161 blood samples of notebook 40 (AGS, CANDLE, SAVI, UAID). 20 patients have more than one sample; samples of one patient are not
independent.

**Method.** limma-voom (Law CW, Chen Y, Shi W, Smyth GK. voom: precision weights unlock linear model analysis
tools for RNA-seq read counts. *Genome Biol* 2014;15:R29), with the patient as a blocking factor:
`duplicateCorrelation` estimates how alike two samples of the same patient are, and the model uses that
correlation (Smyth GK, Michaud J, Scott HS. Use of within-array replicate spots for assessing differential
expression in microarray experiments. *Bioinformatics* 2005;21:2067-2075). DESeq2 has no way to model a
patient block, so it is not used here.

**Design.** One coefficient for each diagnosis and one for each batch but the first:
`~ 0 + diagnosis + batch`.

**Comparisons.** Every pair of AGS, CANDLE, SAVI and UAID: 6 comparisons.

**Top genes.** Benjamini-Hochberg adjusted p < 0.05 and a fold change of at least 2 in either direction
(|log2 fold change| >= 1).

## Read the counts and the sample table

In [1]:
source("../src/paths.R")
suppressMessages({library(limma); library(edgeR)})
x <- readRDS(art("CANDLE_SAVI", "counts_raw.rds")); s <- readRDS(art("CANDLE_SAVI", "samples.rds"))
stopifnot(identical(colnames(x$counts), s$sample))
s$diagnosis <- factor(make.names(s$diagnosis)); s$batch <- factor(make.names(s$batch))
table(s$diagnosis)
c(samples = nrow(s), patients = length(unique(s$patient)), genes = nrow(x$counts))


   AGS CANDLE   SAVI   UAID 
    16     75     21     49 

samples patients    genes 
     161       78    57789

**Explanation**
- `library(limma)` and `library(edgeR)` (Robinson MD, McCarthy DJ, Smyth GK. *Bioinformatics* 2010;26:139-140)
  provide the methods.
- `stopifnot(identical(...))` checks that the counts and the sample table list the samples in the same order.
- `make.names()` turns labels into names R can use as coefficients: `NIAMS-Batch1` becomes `NIAMS.Batch1`.
  `factor()` makes each label a group.

**Result.** 161 samples from 78 patients; 57,789 genes.

## Design and gene filter

In [2]:
design <- model.matrix(~ 0 + diagnosis + batch, s)
colnames(design) <- sub("^diagnosis", "", colnames(design))
dge  <- DGEList(x$counts, genes = x$genes)
keep <- filterByExpr(dge, design)
dge  <- calcNormFactors(dge[keep, , keep.lib.sizes = FALSE])
c(coefficients = ncol(design), genes_kept = sum(keep))

coefficients   genes_kept 
           9        28757

**Explanation**
- `model.matrix(~ 0 + diagnosis + batch, s)` builds the design: `0 +` gives one column per diagnosis (its mean)
  instead of an intercept, and the batch columns give each batch's shift from the first batch.
  `sub("^diagnosis", "", ...)` shortens the column names, for example `diagnosisSAVI` to `SAVI`.
- `DGEList()` holds the counts and gene names; `filterByExpr(dge, design)` keeps genes with enough counts
  for this design (Chen Y, Lun ATL, Smyth GK. *F1000Research* 2016;5:1438).
- `calcNormFactors()` computes TMM normalization factors (Robinson MD, Oshlack A. *Genome Biol*
  2010;11:R25).

**Result.** 9 coefficients (4 diagnoses, 5 batches); 28,757 genes pass the filter.

## voom with the patient as a block

In [3]:
v   <- voom(dge, design)
cor1 <- duplicateCorrelation(v, design, block = s$patient)$consensus
v   <- voom(dge, design, block = s$patient, correlation = cor1)
cor2 <- duplicateCorrelation(v, design, block = s$patient)$consensus
fit <- lmFit(v, design, block = s$patient, correlation = cor2)
round(c(first_estimate = cor1, second_estimate = cor2), 3)

first_estimate second_estimate 
          0.209           0.209

**Explanation**
- `voom(dge, design)` turns the counts into log2 counts per million and gives each value a precision weight.
- `duplicateCorrelation(v, design, block = s$patient)` estimates the correlation between samples of the same
  patient; `$consensus` is one value for all genes.
- voom and `duplicateCorrelation` are run twice, the second time with the first correlation, so that the
  weights and the correlation agree, as the limma authors recommend (limma User's Guide,
  Bioconductor).
- `lmFit(v, design, block = s$patient, correlation = cor2)` fits each gene's model with the patient block.

**Result.** The correlation between samples of the same patient is 0.209 in both estimates.

## The six comparisons

In [4]:
groups <- c("AGS", "CANDLE", "SAVI", "UAID")
pairs  <- combn(groups, 2, simplify = FALSE)
cm <- makeContrasts(contrasts = sapply(pairs, function(p) paste0(p[2], "-", p[1])), levels = design)
fit2 <- eBayes(contrasts.fit(fit, cm))
tabs <- lapply(setNames(colnames(cm), colnames(cm)), function(k) topTable(fit2, coef = k, number = Inf, sort.by = "P"))
data.frame(comparison = names(tabs),
           adj_p_below_0.05 = sapply(tabs, function(t) sum(t$adj.P.Val < 0.05)),
           top_genes_up = sapply(tabs, function(t) sum(t$adj.P.Val < 0.05 & t$logFC >= 1)),
           top_genes_down = sapply(tabs, function(t) sum(t$adj.P.Val < 0.05 & t$logFC <= -1)), row.names = NULL)

comparison,adj_p_below_0.05,top_genes_up,top_genes_down
<chr>,<int>,<int>,<int>
CANDLE-AGS,3081,804,245
SAVI-AGS,967,495,50
UAID-AGS,849,293,55
SAVI-CANDLE,257,98,41
UAID-CANDLE,76,17,15
UAID-SAVI,12,5,5


**Explanation**
- `combn(groups, 2)` lists every pair of the four diagnoses: 6 pairs.
- `makeContrasts()` writes each comparison as the difference of two diagnosis means, for example
  `CANDLE-AGS`: a positive log2 fold change means higher in the first-named diagnosis.
- `contrasts.fit()` and `eBayes()` give, for each comparison and gene, the log2 fold change, a moderated t
  statistic and its p value; `eBayes()` moves each gene's variance toward the average over all genes
  (empirical Bayes; Smyth GK. *Stat Appl Genet Mol Biol* 2004;3:Article3).
- `topTable(..., number = Inf, sort.by = "P")` returns all genes for one comparison, sorted by p, with the
  Benjamini-Hochberg adjusted p (`adj.P.Val`).
- The table counts, per comparison, the genes with adjusted p < 0.05, and among them those with a fold change
  of at least 2 up (`logFC >= 1`) or down (`logFC <= -1`).

**Result.**

| comparison | adjusted p < 0.05 | top genes, up | top genes, down |
|---|---|---|---|
| CANDLE against AGS | 3,081 | 804 | 245 |
| SAVI against AGS | 967 | 495 | 50 |
| UAID against AGS | 849 | 293 | 55 |
| SAVI against CANDLE | 257 | 98 | 41 |
| UAID against CANDLE | 76 | 17 | 15 |
| UAID against SAVI | 12 | 5 | 5 |

"Up" means higher in the first-named diagnosis.

## The top genes of each comparison

In [5]:
top <- lapply(tabs, function(t) t[t$adj.P.Val < 0.05 & abs(t$logFC) >= 1, c("gene_id", "gene_name", "logFC", "AveExpr", "adj.P.Val")])
for (k in names(tabs)) {
  write.csv(tabs[[k]], art("CANDLE_SAVI", sprintf("de_%s.csv", k)), row.names = FALSE)
  write.csv(top[[k]],  art("CANDLE_SAVI", sprintf("top_genes_%s.csv", k)), row.names = FALSE)
}
for (k in names(top)) { cat("\n", k, ": ", nrow(top[[k]]), " top genes; the 10 with the smallest adjusted p\n", sep = "")
  print(head(transform(top[[k]], logFC = round(logFC, 2), AveExpr = round(AveExpr, 1), adj.P.Val = signif(adj.P.Val, 2)), 10), row.names = FALSE) }
c(genes_in_any_top_list = length(unique(unlist(lapply(top, `[[`, "gene_id")))))


CANDLE-AGS: 1049 top genes; the 10 with the smallest adjusted p
         gene_id      gene_name logFC AveExpr adj.P.Val
 ENSG00000154451           GBP5  3.46     8.7   5.4e-11
 ENSG00000203814      HIST2H2BF  2.58     5.2   7.3e-09
 ENSG00000026103            FAS  1.78     5.6   2.4e-08
 ENSG00000117228           GBP1  2.64     7.7   2.4e-08
 ENSG00000120217          CD274  3.33     5.1   2.8e-08
 ENSG00000240929      HIST2H2BB  2.92     2.3   3.1e-08
 ENSG00000198019         FCGR1B  2.85     3.8   5.5e-08
 ENSG00000237568   RP4-620F22.2  3.65     6.3   5.5e-08
 ENSG00000117226           GBP3  1.73     5.3   5.7e-08
 ENSG00000242663 RP11-196G18.21  2.55     4.9   5.7e-08

SAVI-AGS: 545 top genes; the 10 with the smallest adjusted p
         gene_id gene_name logFC AveExpr adj.P.Val
 ENSG00000100505     TRIM9  2.44     1.3   4.9e-07
 ENSG00000197846 HIST1H2BF  2.22     1.0   4.8e-04
 ENSG00000242550 SERPINB10  2.55     2.0   7.3e-04
 ENSG00000051620     HEBP2  1.02     5.8   7.3e-04
 E

genes_in_any_top_list 
                 1413

**Explanation**
- `top` keeps, for each comparison, the genes with adjusted p < 0.05 and |log2 fold change| >= 1, with their
  identifier, name, log2 fold change, average log2 expression and adjusted p.
- The full table and the top genes of each comparison are written as CSV files, for notebook 42.
- The loop prints, for each comparison, the number of top genes and the 10 with the smallest adjusted p.
- The last line counts the genes in any of the six top lists.

**Result.** 1,413 genes are in at least one of the six top lists. In CANDLE against AGS the genes with the
smallest adjusted p include GBP5, GBP1, GBP3, CD274, FAS and FCGR1B, all higher in CANDLE. Histone genes
(HIST2H2BF, HIST1H2BF, HIST2H2BB, HIST1H2BG) are among the first ten in the three comparisons against AGS. In
SAVI against AGS and SAVI against CANDLE, TRIM9 has the smallest adjusted p, higher in SAVI.

## References

The methods, gene lists and software used in this notebook.

1. R Core Team. *R: A Language and Environment for Statistical Computing*. R Foundation for Statistical Computing, Vienna, Austria.
2. Ritchie ME, Phipson B, Wu D, Hu Y, Law CW, Shi W, Smyth GK. limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Res* 2015;43:e47.
3. Smyth GK et al. *limma: Linear Models for Microarray and RNA-Seq Data. User's Guide*. Bioconductor.
4. Robinson MD, McCarthy DJ, Smyth GK. edgeR: a Bioconductor package for differential expression analysis of digital gene expression data. *Bioinformatics* 2010;26:139-140.
5. Robinson MD, Oshlack A. A scaling normalization method for differential expression analysis of RNA-seq data. *Genome Biol* 2010;11:R25.
6. Chen Y, Lun ATL, Smyth GK. From reads to genes to pathways: differential expression analysis of RNA-Seq experiments using Rsubread and the edgeR quasi-likelihood pipeline. *F1000Research* 2016;5:1438.
7. Law CW, Chen Y, Shi W, Smyth GK. voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biol* 2014;15:R29.
8. Smyth GK, Michaud J, Scott HS. Use of within-array replicate spots for assessing differential expression in microarray experiments. *Bioinformatics* 2005;21:2067-2075.
9. Smyth GK. Linear models and empirical Bayes methods for assessing differential expression in microarray experiments. *Stat Appl Genet Mol Biol* 2004;3:Article3.
10. Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. *J R Stat Soc B* 1995;57:289-300.
11. Love MI, Anders S, Kim V, Huber W. RNA-seq workflow: gene-level exploratory analysis and differential expression. *F1000Research* 2015;4:1070.